# Daily realized variance for USD/JPY, aligned to FRED DEXJPUS

**Data sources**
- **Daily:** FRED series `DEXJPUS` (Japanese yen per US dollar, noon buying rate in New York; source: Board of Governors of the Federal Reserve System, H.10 release). It's downloaded from `https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXJPUS`.
- **Intraday:** FRED has no intraday FX data, so intraday USDJPY bars (M1 or M5) come from your own forexsb.com export.

**Method:** 5-minute log returns × 100 (percent, same units as the GARCH inputs). Each return is assigned to the FRED date whose noon-ET fixing ends its window, so RV_t covers the same (noon_{t-1}, noon_t] interval as the daily return r_t. RV_t = Σ r_i² in %² per day, directly comparable to the GARCH σ².

In [1]:
import io
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- settings: edit these ----------------
INTRADAY_FILE = "USDJPY_M5.csv"   # forexsb export (M1 or M5)
FRED_FILE     = None              # None = download from FRED; or a path to a saved DEXJPUS.csv
SOURCE_TZ     = "UTC"             # time zone of the intraday timestamps (check your export!)
BAR_TIME      = "open"            # "open" if timestamps mark bar open (forexsb/MT4 default), else "close"
MINUTES       = 5                 # sampling interval for RV
MIN_RETURNS   = 200               # flag days with fewer returns (288 = full 24h of 5-min returns)
OUT_FILE      = "rv_usdjpy.csv"

FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXJPUS"
NY = "America/New_York"

## 1. Daily DEXJPUS from FRED

In [2]:
def load_fred(path=None):
    """Return a Series of DEXJPUS indexed by date (missing values dropped)."""
    if path:
        raw = open(path, "rb").read()
    else:
        with urllib.request.urlopen(FRED_URL, timeout=30) as resp:
            raw = resp.read()
    df = pd.read_csv(io.BytesIO(raw), na_values=".")
    date_col = df.columns[0]  # "observation_date" (current) or "DATE" (older files)
    df[date_col] = pd.to_datetime(df[date_col])
    s = df.set_index(date_col)["DEXJPUS"].astype(float).dropna()
    s.index.name = "date"
    return s

fred = load_fred(FRED_FILE)
fred.tail()

date
2026-11-06    158.23
2026-11-09    157.99
2026-11-10    158.76
2026-11-12    159.60
2026-11-13    159.93
Name: DEXJPUS, dtype: float64

## 2. Intraday USDJPY bars → 5-minute closes (New York time)

In [3]:
def load_intraday(path, source_tz="UTC", bar_time="open"):
    """
    Read an intraday OHLC CSV and return close prices indexed by the time each
    close was observed, in New York time.

    Accepts the common layouts:
      Time,Open,High,Low,Close,Volume              (one datetime column)
      Date,Time,Open,High,Low,Close,Volume         (separate date and time)
      MT4 style without header: 2024.01.02,00:00,o,h,l,c,v
    """
    df = pd.read_csv(path, sep=None, engine="python")
    cols_lower = [str(c).strip().lower() for c in df.columns]

    if "close" not in cols_lower:  # no header -> MT4 layout
        df = pd.read_csv(path, sep=None, engine="python", header=None)
        if df.shape[1] >= 7:
            df = df.iloc[:, :7]
            df.columns = ["date", "time", "open", "high", "low", "close", "volume"]
        else:
            df = df.iloc[:, :6]
            df.columns = ["time", "open", "high", "low", "close", "volume"]
    else:
        df.columns = cols_lower

    if "date" in df.columns and "time" in df.columns:
        stamp = df["date"].astype(str) + " " + df["time"].astype(str)
    else:
        time_col = next(c for c in df.columns if c in ("time", "datetime", "timestamp", "date"))
        stamp = df[time_col].astype(str)
    stamp = stamp.str.replace(".", "-", n=2, regex=False)  # 2024.01.02 -> 2024-01-02

    ts = pd.to_datetime(stamp)
    close = pd.Series(df["close"].astype(float).values, index=ts).sort_index()
    close = close[~close.index.duplicated(keep="last")]

    if bar_time == "open":  # shift to the moment the close price was observed
        step = close.index.to_series().diff().mode().iloc[0]
        close.index = close.index + step

    close.index = close.index.tz_localize(source_tz, ambiguous="NaT",
                                          nonexistent="shift_forward").tz_convert(NY)
    return close[close.index.notna()]


def to_5min(close, minutes=5):
    """Last close in each 5-minute bucket (works for M1 or M5 input)."""
    return close.resample(f"{minutes}min", label="right", closed="right").last().dropna()


close_5m = to_5min(load_intraday(INTRADAY_FILE, SOURCE_TZ, BAR_TIME), MINUTES)

# keep only FRED dates that overlap the intraday sample
start = close_5m.index[0].tz_localize(None).normalize() - pd.Timedelta(days=7)
end   = close_5m.index[-1].tz_localize(None).normalize() + pd.Timedelta(days=7)
fred = fred[(fred.index >= start) & (fred.index <= end)]
close_5m.head()

FileNotFoundError: [Errno 2] No such file or directory: 'USDJPY_M5.csv'

## 3. Realized variance on FRED's noon-to-noon days

In [ ]:
def realized_variance(close_5m, fred, max_gap_hours=72):
    """
    Daily RV aligned to FRED noon fixings.

    Each 5-min log return (x100) is assigned to the first FRED date whose
    noon-ET timestamp is >= the return's end time. Weekends and holidays fold
    into the next FRED observation, exactly like the daily return.
    Returns spanning a data gap longer than max_gap_hours are dropped.
    """
    r = 100 * np.log(close_5m).diff()
    gap = close_5m.index.to_series().diff()
    r = r[gap <= pd.Timedelta(hours=max_gap_hours)].dropna()

    noon = (pd.DatetimeIndex(fred.index) + pd.Timedelta(hours=12)).tz_localize(NY)
    pos = np.searchsorted(noon.values, r.index.values, side="left")
    inside = pos < len(noon)          # drop returns after the last FRED fixing
    r, pos = r[inside], pos[inside]

    keep = pos > 0                    # first FRED date has no previous noon -> incomplete window
    r, pos = r[keep], pos[keep]

    day = fred.index[pos]
    out = pd.DataFrame({"r": r.values, "date": day})
    g = out.groupby("date")["r"]
    res = pd.DataFrame({
        "n_returns": g.size(),
        "RV": g.apply(lambda x: np.sum(x ** 2)),     # %^2 per day
        "sum_intraday_ret": g.sum(),                  # ~ daily log return
    })
    res["RV_vol_ann"] = np.sqrt(res["RV"] * 252)      # annualized %, like the GARCH plot

    daily_ret = 100 * np.log(fred).diff()
    res["fred_ret"] = daily_ret.reindex(res.index)
    res["fred_ret_sq"] = res["fred_ret"] ** 2
    return res


rv = realized_variance(close_5m, fred)
rv["low_coverage"] = rv["n_returns"] < MIN_RETURNS
rv.to_csv(OUT_FILE)
rv.head()

## 4. Sanity checks
If the ratio is far from 1 or the correlation is well below 0.95, the time zone (`SOURCE_TZ`) or `BAR_TIME` setting is probably wrong.

In [ ]:
good = rv[~rv["low_coverage"]].dropna()
print(f"days: {len(rv)}  (low coverage: {rv['low_coverage'].sum()})")
print(f"mean RV        : {good['RV'].mean():.4f} %^2/day")
print(f"mean FRED r^2  : {good['fred_ret_sq'].mean():.4f} %^2/day")
print(f"ratio RV / r^2 : {good['RV'].mean() / good['fred_ret_sq'].mean():.2f}  (roughly 0.7-1.3 expected)")
print(f"corr(sum intraday ret, FRED ret): {good['sum_intraday_ret'].corr(good['fred_ret']):.3f}  (~0.95+ expected)")

## 5. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(good.index, good["RV_vol_ann"], lw=0.8, label="Daily 5-min realized vol (annualized)")
ax.plot(good.index, np.sqrt(good["fred_ret_sq"] * 252), lw=0.5, alpha=0.5,
        label="|daily FRED return| (annualized)")
ax.set_ylabel("Annualized volatility (%)")
ax.set_title("USD/JPY: daily realized volatility vs single-day squared return")
ax.legend()
ax.grid(alpha=0.3)
plt.show()